# Sampled focus-rule smoke test

Automatic, unblinded inspection of 20 sampled documents. No reference labels or accuracy estimates. Literal foci are not exhaustive predicate inventories. B2 was not changed after this draw.


In [ ]:
from pathlib import Path
import json
import hashlib
import pandas as pd
from IPython.display import display, Markdown
current = Path.cwd()
ROOT = next((p for p in [current, *current.parents] if (p / "reports/sampled_focus_smoke_v1.json").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Start from the project directory or its notebooks folder.")
report = json.loads((ROOT / "reports/sampled_focus_smoke_v1.json").read_text())
corpus_path = ROOT / "data/processed/model_corpus_v1.csv"
if hashlib.sha256(corpus_path.read_bytes()).hexdigest() != report["corpus_sha256"]:
    raise ValueError("Frozen corpus hash differs.")
corpus = pd.read_csv(corpus_path, keep_default_na=False).set_index("execution_number")
predictions = pd.DataFrame(report["predictions"])
documents = pd.DataFrame(report["documents"])
display(documents[["execution_number", "stratum", "literal_focus_count"]])


In [ ]:
display(pd.crosstab(predictions["stratum"], predictions["prediction"]))
print("Prediction counts only: no reference labels, no accuracy or recall.")
print("Documents with no retrieved focus:")
display(documents.loc[documents.literal_focus_count.eq(0), ["execution_number", "stratum"]])


## Inspect one document

Change the execution number below to any sampled ID. Complete context is shown alongside rule outputs. These are predictions, not owner-confirmed labels. Reading this view is unblinded.


In [ ]:
def show_document(execution_number):
    if execution_number not in set(documents.execution_number):
        raise ValueError("Choose an ID from this fixed sample.")
    row = corpus.loc[execution_number]
    display(Markdown(f"### Execution {execution_number}\n\n[Source statement]({row.source_url})"))
    display(Markdown(row.model_text))
    cols = ["focus_start", "focus_end", "predicate_focus", "prediction", "reason"]
    display(predictions.loc[predictions.execution_number.eq(execution_number), cols])
show_document(378)
